In [1]:
from pathlib import Path
from time import perf_counter
import json

import numpy as np
import pandas as pd
import duckdb

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    average_precision_score,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"

con = duckdb.connect()

training_data = con.read_parquet(
    str(PROCESSED_DIR / "historical_combined_features_20161130.parquet")
).df()

evaluation_data = con.read_parquet(
    str(PROCESSED_DIR / "later_combined_features_20170228.parquet")
).df()

feature_config = json.loads(
    (REPORT_DIR / "combined_feature_config.json").read_text(
        encoding="utf-8"
    )
)

transaction_features = [
    "transaction_rows_90d",
    "transaction_days_90d",
    "cancel_rows_90d",
    "auto_renew_share_90d",
    "avg_plan_days_90d",
    "days_since_last_transaction_90d",
    "has_transaction_90d",
]

combined_features = feature_config["features"]

assert len(transaction_features) == 7
assert len(combined_features) == 23
assert set(transaction_features).issubset(combined_features)

assert training_data["msno"].is_unique
assert evaluation_data["msno"].is_unique

y_train = training_data["is_churn"].astype("int8")
y_eval = evaluation_data["is_churn"].astype("int8")

assert y_train.isin([0, 1]).all()
assert y_eval.isin([0, 1]).all()

print("Training users:", len(training_data))
print("Evaluation users:", len(evaluation_data))

Training users: 882127
Evaluation users: 970960


## Define Fixed Model Configurations

In [2]:
models = {
    "Logistic regression": Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median", add_indicator=True),
        ),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(max_iter=1000, random_state=42),
        ),
    ]),
    "XGBoost": XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=100,
        tree_method="hist",
        random_state=42,
        n_jobs=4,
    ),
    "LightGBM": LGBMClassifier(
        objective="binary",
        n_estimators=100,
        random_state=42,
        n_jobs=4,
    ),
}

## Compare Transaction-Only and Combined Features

Use identical cohorts and model configurations for both feature sets.
Fit preprocessing on historical training data only.

In [3]:
feature_sets = {
    "Transaction only": transaction_features,
    "Transaction + listening": combined_features,
}

evaluation_rows = []
fitted_models = {}
evaluation_predictions = {}

for feature_set_name, columns in feature_sets.items():
    X_train = training_data[columns].astype("float64")
    X_eval = evaluation_data[columns].astype("float64")

    assert not np.isinf(X_train.to_numpy()).any()
    assert not np.isinf(X_eval.to_numpy()).any()

    for model_name, model_template in models.items():
        print(
            f"Training {model_name} | {feature_set_name}...",
            flush=True,
        )

        model = clone(model_template)

        start = perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = perf_counter() - start

        probabilities = model.predict_proba(X_eval)[:, 1]

        assert np.isfinite(probabilities).all()
        assert (
            (probabilities >= 0)
            & (probabilities <= 1)
        ).all()

        key = (feature_set_name, model_name)
        fitted_models[key] = model
        evaluation_predictions[key] = probabilities

        evaluation_rows.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "feature_count": len(columns),
            "log_loss": log_loss(y_eval, probabilities),
            "roc_auc": roc_auc_score(y_eval, probabilities),
            "average_precision": average_precision_score(
                y_eval, probabilities
            ),
            "fit_seconds": fit_seconds,
        })

        print(f"Finished in {fit_seconds:.1f} seconds.")

behavior_model_results = (
    pd.DataFrame(evaluation_rows)
    .sort_values(["model", "feature_set"])
    .reset_index(drop=True)
)

behavior_model_results

Training Logistic regression | Transaction only...
Finished in 0.7 seconds.
Training XGBoost | Transaction only...
Finished in 1.3 seconds.
Training LightGBM | Transaction only...
[LightGBM] [Info] Number of positive: 85237, number of negative: 796890
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011190 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 159
[LightGBM] [Info] Number of data points in the train set: 882127, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.096627 -> initscore=-2.235281
[LightGBM] [Info] Start training from score -2.235281
Finished in 1.2 seconds.
Training Logistic regression | Transaction + listening...
Finished in 2.9 seconds.
Training XGBoost | Transaction + listening...
Finished in 2.1 seconds.
Training LightGBM | Transaction + listening...
[LightGBM] [Info] Number of positive

,feature_set,model,feature_count,log_loss,roc_auc,average_precision,fit_seconds
0,Transaction + listening,LightGBM,23,0.225507,0.839498,0.494052,1.826973
1,Transaction only,LightGBM,7,0.229887,0.829479,0.488706,1.245423
2,Transaction + listening,Logistic regression,23,0.207310,0.849893,0.550063,2.928695
3,Transaction only,Logistic regression,7,0.212926,0.829580,0.536523,0.749575
4,Transaction + listening,XGBoost,23,0.230022,0.839235,0.489641,2.086164
5,Transaction only,XGBoost,7,0.229378,0.828912,0.490584,1.253287


In [4]:
metric_columns = [
    "model",
    "log_loss",
    "roc_auc",
    "average_precision",
]

transaction_results = behavior_model_results.loc[
    behavior_model_results["feature_set"] == "Transaction only",
    metric_columns,
]

combined_results = behavior_model_results.loc[
    behavior_model_results["feature_set"] == "Transaction + listening",
    metric_columns,
]

behavior_comparison = transaction_results.merge(
    combined_results,
    on="model",
    validate="one_to_one",
    suffixes=("_transaction", "_combined"),
)

behavior_comparison["log_loss_improvement_pct"] = (
    100
    * (
        behavior_comparison["log_loss_transaction"]
        - behavior_comparison["log_loss_combined"]
    )
    / behavior_comparison["log_loss_transaction"]
)

behavior_comparison[
    [
        "model",
        "log_loss_transaction",
        "log_loss_combined",
        "log_loss_improvement_pct",
        "roc_auc_combined",
        "average_precision_combined",
    ]
]

,model,log_loss_transaction,log_loss_combined,log_loss_improvement_pct,roc_auc_combined,average_precision_combined
0,LightGBM,0.229887,0.225507,1.905423,0.839498,0.494052
1,Logistic regression,0.212926,0.207310,2.637662,0.849893,0.550063
2,XGBoost,0.229378,0.230022,-0.280687,0.839235,0.489641


## Listening Duration Policy Sensitivity

Compare predeclared 12-hour, 24-hour, and 48-hour duration bounds,
plus a model excluding duration-derived features.
Keep cohorts and logistic regression settings fixed.

In [5]:
duration_columns = [
    "valid_listening_seconds_30d",
    "valid_duration_days_30d",
    "flagged_duration_days_30d",
    "avg_seconds_per_valid_day_30d",
]

logs_pattern = str(
    PROCESSED_DIR / "listening_windows_30d" / "**" / "*.parquet"
)

temp_dir = PROJECT_ROOT / "data" / "duckdb_tmp"
temp_dir.mkdir(parents=True, exist_ok=True)

sensitivity_con = duckdb.connect(config={
    "threads": 2,
    "memory_limit": "2GB",
    "temp_directory": str(temp_dir),
    "max_temp_directory_size": "2GB",
})

duration_sql = """
WITH cleaned AS (
    SELECT
        msno,
        CASE
            WHEN ISFINITE(total_secs_raw)
             AND total_secs_raw BETWEEN 0 AND ?
            THEN total_secs_raw
            ELSE NULL
        END AS listening_seconds
    FROM read_parquet(?, hive_partitioning = true)
    WHERE cohort = ?
)
SELECT
    msno,
    SUM(listening_seconds) AS valid_listening_seconds_30d,
    COUNT(listening_seconds) AS valid_duration_days_30d,
    COUNT(*) - COUNT(listening_seconds) AS flagged_duration_days_30d,
    AVG(listening_seconds) AS avg_seconds_per_valid_day_30d
FROM cleaned
GROUP BY msno
"""

def replace_duration_features(dataset, cohort, upper_bound):
    duration_features = sensitivity_con.execute(
        duration_sql,
        [upper_bound, logs_pattern, cohort],
    ).df()

    updated = dataset.drop(columns=duration_columns).merge(
        duration_features,
        on="msno",
        how="left",
        validate="one_to_one",
    )

    for column in [
        "valid_duration_days_30d",
        "flagged_duration_days_30d",
    ]:
        updated[column] = updated[column].fillna(0).astype("int64")

    assert updated["msno"].equals(dataset["msno"])
    assert updated["is_churn"].equals(dataset["is_churn"])

    assert (
        updated["valid_duration_days_30d"]
        + updated["flagged_duration_days_30d"]
        == updated["recorded_days_30d"]
    ).all()

    return updated

In [6]:
def score_duration_policy(policy, probabilities, feature_count):
    return {
        "duration_policy": policy,
        "feature_count": feature_count,
        "log_loss": log_loss(y_eval, probabilities),
        "roc_auc": roc_auc_score(y_eval, probabilities),
        "average_precision": average_precision_score(
            y_eval, probabilities
        ),
    }

baseline_probabilities = evaluation_predictions[
    ("Transaction + listening", "Logistic regression")
]

sensitivity_rows = [
    score_duration_policy(
        "24-hour bound",
        baseline_probabilities,
        len(combined_features),
    )
]

for hours in [12, 48]:
    print(f"Evaluating the {hours}-hour bound...", flush=True)

    policy_train = replace_duration_features(
        training_data,
        "historical_train",
        hours * 3600,
    )

    policy_eval = replace_duration_features(
        evaluation_data,
        "later_evaluation",
        hours * 3600,
    )

    policy_model = clone(models["Logistic regression"])

    policy_model.fit(
        policy_train[combined_features].astype("float64"),
        y_train,
    )

    probabilities = policy_model.predict_proba(
        policy_eval[combined_features].astype("float64")
    )[:, 1]

    assert np.isfinite(probabilities).all()

    sensitivity_rows.append(
        score_duration_policy(
            f"{hours}-hour bound",
            probabilities,
            len(combined_features),
        )
    )

    del policy_train, policy_eval, policy_model

no_duration_features = [
    column
    for column in combined_features
    if column not in duration_columns
]

no_duration_model = clone(models["Logistic regression"])

no_duration_model.fit(
    training_data[no_duration_features].astype("float64"),
    y_train,
)

no_duration_probabilities = no_duration_model.predict_proba(
    evaluation_data[no_duration_features].astype("float64")
)[:, 1]

assert np.isfinite(no_duration_probabilities).all()

sensitivity_rows.append(
    score_duration_policy(
        "No duration-derived features",
        no_duration_probabilities,
        len(no_duration_features),
    )
)

duration_sensitivity_results = pd.DataFrame(sensitivity_rows)
duration_sensitivity_results

Evaluating the 12-hour bound...
Evaluating the 48-hour bound...


,duration_policy,feature_count,log_loss,roc_auc,average_precision
0,24-hour bound,23,0.207310,0.849893,0.550063
1,12-hour bound,23,0.207282,0.849990,0.550105
2,48-hour bound,23,0.207319,0.849879,0.550083
3,No duration-derived features,19,0.207280,0.849975,0.550125


## Evaluate the Highest-Risk 10% of Users

Measure churn concentration and coverage under a fixed contact budget.
These metrics describe risk targeting, not intervention effectiveness.

In [7]:
def evaluate_top_fraction(y_true, probabilities, fraction=0.1):
    labels = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)

    assert 0 < fraction <= 1
    assert len(labels) == len(probabilities)
    assert np.isfinite(probabilities).all()

    selected_count = int(np.ceil(len(labels) * fraction))

    ranked_indices = np.argsort(
        -probabilities,
        kind="stable",
    )

    selected_indices = ranked_indices[:selected_count]
    selected_churns = int(labels[selected_indices].sum())
    total_churns = int(labels.sum())

    overall_churn_rate = labels.mean()
    precision = selected_churns / selected_count

    return {
        "selected_users": selected_count,
        "selected_churned_users": selected_churns,
        "precision_at_10pct": precision,
        "recall_at_10pct": (
            selected_churns / total_churns
            if total_churns else np.nan
        ),
        "lift_at_10pct": (
            precision / overall_churn_rate
            if overall_churn_rate else np.nan
        ),
    }

targeting_rows = []

for (feature_set, model_name), probabilities in (
    evaluation_predictions.items()
):
    result = evaluate_top_fraction(
        y_eval,
        probabilities,
        fraction=0.1,
    )

    targeting_rows.append({
        "feature_set": feature_set,
        "model": model_name,
        **result,
    })

targeting_results = (
    pd.DataFrame(targeting_rows)
    .sort_values("precision_at_10pct", ascending=False)
    .reset_index(drop=True)
)

targeting_results

,feature_set,model,selected_users,selected_churned_users,precision_at_10pct,recall_at_10pct,lift_at_10pct
0,Transaction + listening,Logistic regression,97096,51950,0.535037,0.594870,5.948700
1,Transaction only,Logistic regression,97096,50129,0.516283,0.574018,5.740181
2,Transaction + listening,LightGBM,97096,48542,0.499938,0.555846,5.558456
3,Transaction + listening,XGBoost,97096,47225,0.486374,0.540765,5.407649
4,Transaction only,XGBoost,97096,46098,0.474767,0.527860,5.278598
5,Transaction only,LightGBM,97096,45977,0.473521,0.526474,5.264743


In [8]:
import platform
import sklearn
import xgboost
import lightgbm

REPORT_DIR.mkdir(parents=True, exist_ok=True)

report_tables = {
    "behavior_model_results": behavior_model_results,
    "behavior_feature_comparison": behavior_comparison,
    "duration_sensitivity_results": duration_sensitivity_results,
    "risk_targeting_results": targeting_results,
}

for name, table in report_tables.items():
    table.to_csv(REPORT_DIR / f"{name}.csv", index=False)

evaluation_config = {
    "evaluation_method": "historical cohort evaluation",
    "training_feature_cutoff": "2016-11-30",
    "training_outcome_end": "2017-01-30",
    "evaluation_feature_cutoff": "2017-02-28",
    "feature_sets": feature_sets,
    "models": {
        name: model.get_params(deep=True)
        for name, model in models.items()
    },
    "hyperparameter_search_in_this_notebook": False,
    "duration_sensitivity_model": "Logistic regression",
    "duration_bounds_seconds": [43200, 86400, 172800],
    "duration_exclusion_checked": True,
    "retained_duration_bound_seconds": 86400,
    "targeting_fraction": 0.1,
    "targeting_tie_break": "stable input row order",
    "independent_test_evaluation_completed": False,
    "versions": {
        "python": platform.python_version(),
        "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
        "lightgbm": lightgbm.__version__,
    },
}

(REPORT_DIR / "behavior_evaluation_config.json").write_text(
    json.dumps(evaluation_config, indent=2, default=str) + "\n",
    encoding="utf-8",
)

print("Behavior evaluation, sensitivity, and targeting reports saved.")

Behavior evaluation, sensitivity, and targeting reports saved.


## Behavioral Feature Evaluation and Risk Targeting

Compared seven transaction features with 23 combined transaction and listening features using identical historical training and later evaluation cohorts, with fixed model configurations.

| Model | Transaction-only Log Loss ↓ | Combined Log Loss ↓ | Combined ROC-AUC ↑ | Combined AP ↑ |
|---|---:|---:|---:|---:|
| Logistic Regression | 0.212926 | **0.207310** | **0.849893** | **0.550063** |
| XGBoost | 0.229378 | 0.230022 | 0.839235 | 0.489641 |
| LightGBM | 0.229887 | 0.225507 | 0.839498 | 0.494052 |

Listening features reduced Logistic Regression Log Loss by 2.64% and LightGBM Log Loss by 1.91%. XGBoost ranking performance improved, but Log Loss slightly worsened.

### Duration Sensitivity

For Logistic Regression, 12-hour, 24-hour, and 48-hour duration bounds, plus exclusion of all duration-derived features, produced Log Loss values between 0.207280 and 0.207319.

Performance was numerically similar across these checks. The predeclared 24-hour policy is retained; this does not establish that the threshold is operationally correct.

### Highest-Risk 10% of Users

The combined-feature Logistic Regression selected 97,096 users, including 51,950 observed churners:

- Precision@10%: 53.50%.
- Recall@10%: 59.49%.
- Lift@10%: 5.95.

At the same selection budget, listening features identified 1,821 additional churners relative to transaction-only Logistic Regression.

These metrics measure risk concentration, not customers retained or causal intervention effects. Results remain exploratory later-cohort evaluations, with the previously documented label and source-data limitations.

See [the evaluation notebook](notebooks/12_behavior_model_evaluation.ipynb), [feature comparison](reports/behavior_feature_comparison.csv), [duration sensitivity](reports/duration_sensitivity_results.csv), and [targeting results](reports/risk_targeting_results.csv).